# Project 03 — Credit Risk: PD / LGD / EAD & Expected Loss
## Notebook 04 — Expected Loss & Portfolio Risk Contribution

This notebook extends the Probability of Default (PD) model from Notebook 02 and the LGD/EAD assumptions documented in Notebook 03. It combines those inputs to estimate expected loss at the loan and portfolio levels, then examines where expected losses are concentrated.

**Objectives**

- Verify the exported loan-level expected-loss calculations.
- Quantify portfolio exposure, expected loss, and expected-loss rate for development and validation samples.
- Identify large individual contributors to expected loss.
- Compare losses across credit grade, loan purpose, PD-based risk band, and borrower income band.

**Methodology:**

$$EL_i = PD_i \times LGD_i \times EAD_i$$

PD is a model prediction, LGD is an assumed credit-grade value, and EAD is proxied by funded loan amount. The results are therefore model-based estimates under simplified assumptions, not forecasts of realized cash losses.

## 1. Load Data and Validate Inputs

The notebook uses the processed development and validation files exported by Notebook 03. Keeping these files in `data/processed/` makes the workflow repeatable and prevents the calculations in Notebook 03 from having to be recreated here.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
data_dir = Path("../data/processed")

train_df = pd.read_csv(data_dir / "credit_risk_development_el.csv")
valid_df = pd.read_csv(data_dir / "credit_risk_validation_el.csv")

print(f"Development loans: {len(train_df):,}")
print(f"Validation loans: {len(valid_df):,}")
train_df.head()

Development loans: 1,495,018
Validation loans: 368,062


,id,grade,annual_inc,purpose,issue_d,default_flag,loan_amnt,funded_amnt,predicted_pd,lgd_assumption,ead_proxy,expected_loss
0,85781,C,40000.0,other,2007-06-01,0,1400.0,1400.0,0.158369,0.5,1400.0,110.858591
1,87023,E,22000.0,debt_consolidation,2007-06-01,0,7500.0,7500.0,0.255729,0.6,7500.0,1150.778947
2,88637,C,20000.0,debt_consolidation,2007-06-01,0,6000.0,6000.0,0.176585,0.5,6000.0,529.754626
3,88046,B,30000.0,debt_consolidation,2007-06-01,0,4400.0,4400.0,0.082347,0.4,4400.0,144.931286
4,85961,B,36000.0,other,2007-06-01,0,1200.0,1200.0,0.088660,0.4,1200.0,42.556829


### 1.1 Input validation

Annual income is not required for expected-loss calculation, so missing income values are retained and handled separately in the income segmentation. The fields used directly in the EL formula must be present and valid.

In [3]:
critical_columns = [
    "id", "grade", "purpose", "funded_amnt", "predicted_pd",
    "lgd_assumption", "ead_proxy", "expected_loss",
]
required_schema_columns = critical_columns + ["annual_inc"]

for name, df in [("development", train_df), ("validation", valid_df)]:
    missing_columns = sorted(set(required_schema_columns) - set(df.columns))
    assert not missing_columns, f"{name}: missing columns {missing_columns}"
    assert df[critical_columns].notna().all().all(), (
        f"{name}: missing values found in critical EL inputs"
    )
    assert df["id"].is_unique, f"{name}: loan IDs are not unique"
    assert df["predicted_pd"].between(0, 1).all(), (
        f"{name}: PD values fall outside [0, 1]"
    )
    assert df["lgd_assumption"].between(0, 1).all(), (
        f"{name}: LGD assumptions fall outside [0, 1]"
    )
    assert (df["ead_proxy"] >= 0).all(), f"{name}: negative EAD proxy found"
    assert (df["expected_loss"] >= 0).all(), f"{name}: negative expected loss found"
    assert df["ead_proxy"].sum() > 0, f"{name}: total EAD must be positive"

print(f"Development loans with missing income: {train_df['annual_inc'].isna().sum():,}")
print(f"Validation loans with missing income: {valid_df['annual_inc'].isna().sum():,}")
print("Critical input validation passed.")

Development loans with missing income: 4
Validation loans with missing income: 0
Critical input validation passed.


## 2. Loan-Level Expected Loss

### 2.1 Recalculate and verify expected loss

Expected loss for each loan is calculated as PD multiplied by LGD and EAD. This section independently recomputes that product and compares it with the `expected_loss` value exported from Notebook 03.

In [4]:
for df in (train_df, valid_df):
    df["expected_loss_check"] = (
        df["predicted_pd"] * df["lgd_assumption"] * df["ead_proxy"]
    )
    df["expected_loss_abs_diff"] = (
        df["expected_loss"] - df["expected_loss_check"]
    ).abs()

train_max_diff = train_df["expected_loss_abs_diff"].max()
valid_max_diff = valid_df["expected_loss_abs_diff"].max()

print(f"Development maximum absolute difference: {train_max_diff:.3e}")
print(f"Validation maximum absolute difference: {valid_max_diff:.3e}")

assert np.allclose(
    train_df["expected_loss"], train_df["expected_loss_check"],
    rtol=1e-6, atol=1e-6,
)
assert np.allclose(
    valid_df["expected_loss"], valid_df["expected_loss_check"],
    rtol=1e-6, atol=1e-6,
)

print("Loan-level expected-loss verification passed.")

Development maximum absolute difference: 3.638e-12
Validation maximum absolute difference: 3.638e-12
Loan-level expected-loss verification passed.


In [5]:

train_df[[
    "id", "grade", "funded_amnt", "predicted_pd",
    "lgd_assumption", "ead_proxy", "expected_loss",
]].head(10).style.format({
    "funded_amnt": "${:,.2f}",
    "predicted_pd": "{:.2%}",
    "lgd_assumption": "{:.1%}",
    "ead_proxy": "${:,.2f}",
    "expected_loss": "${:,.2f}",
})


,id,grade,funded_amnt,predicted_pd,lgd_assumption,ead_proxy,expected_loss
0,85781,C,"$1,400.00",15.84%,50.0%,"$1,400.00",$110.86
1,87023,E,"$7,500.00",25.57%,60.0%,"$7,500.00","$1,150.78"
2,88637,C,"$6,000.00",17.66%,50.0%,"$6,000.00",$529.75
3,88046,B,"$4,400.00",8.23%,40.0%,"$4,400.00",$144.93
4,85961,B,"$1,200.00",8.87%,40.0%,"$1,200.00",$42.56
5,85818,C,"$5,000.00",16.37%,50.0%,"$5,000.00",$409.28
6,85675,E,"$1,000.00",25.66%,60.0%,"$1,000.00",$153.98
7,84670,A,"$5,000.00",2.37%,40.0%,"$5,000.00",$47.44
8,84098,A,"$2,500.00",3.01%,40.0%,"$2,500.00",$30.11
9,83979,A,"$3,000.00",2.60%,40.0%,"$3,000.00",$31.22


**Interpretation**

The exported calculations and the independent PD × LGD × EAD recomputation agree to numerical precision in both samples. This confirms internal calculation consistency; it does not, by itself, validate the predictive accuracy of the PD model or the LGD/EAD assumptions.

The displayed rows are a sample of loan-level records, not the highest-risk loans. The next section aggregates all loans to the portfolio level.

## 3. Portfolio Expected Loss

### 3.1 Portfolio-level measures

For each sample, the notebook calculates loan count, total EAD proxy, total expected loss, and expected-loss rate:

$$EL\ Rate = \frac{\sum_i EL_i}{\sum_i EAD_i}$$

The denominator is total EAD, so this is an exposure-weighted portfolio loss rate rather than the simple average of individual loan loss rates.

In [6]:
def calculate_portfolio_summary(df):
    total_ead = df["ead_proxy"].sum()
    total_expected_loss = df["expected_loss"].sum()
    return {
        "Number of Loans": len(df),
        "Total EAD ($)": total_ead,
        "Total Expected Loss ($)": total_expected_loss,
        "Expected Loss Rate (%)": (
            total_expected_loss / total_ead * 100 if total_ead > 0 else np.nan
        ),
    }

portfolio_summary = pd.DataFrame({
    "Development": calculate_portfolio_summary(train_df),
    "Validation": calculate_portfolio_summary(valid_df),
}).T

portfolio_summary.style.format({
    "Number of Loans": "{:,.0f}",
    "Total EAD ($)": "${:,.2f}",
    "Total Expected Loss ($)": "${:,.2f}",
    "Expected Loss Rate (%)": "{:.3f}%",
})

,Number of Loans,Total EAD ($),Total Expected Loss ($),Expected Loss Rate (%)
Development,"1,495,018","$21,595,123,875.00","$2,200,968,576.86",10.192%
Validation,"368,062","$5,556,844,900.00","$535,593,371.63",9.638%


In [7]:
# Reconcile portfolio totals to independently recalculated loan-level EL.
for name, df in [("development", train_df), ("validation", valid_df)]:
    assert np.isclose(
        df["expected_loss"].sum(), df["expected_loss_check"].sum(),
        rtol=1e-10, atol=1e-6,
    ), f"{name}: portfolio EL does not reconcile to the loan-level check"
    assert df["ead_proxy"].sum() > 0, f"{name}: total EAD must be positive"

print("Portfolio aggregation checks passed.")

Portfolio aggregation checks passed.


### 3.2 Interpretation

The development sample contains **1,495,018 loans**, total EAD proxy of approximately **$21.60 billion**, and total expected loss of **$2.201 billion**, producing an expected-loss rate of **10.192%**. The validation sample contains **368,062 loans**, total EAD proxy of approximately **$5.557 billion**, and expected loss of **$535.59 million**, producing an expected-loss rate of **9.638%**.

The development rate is about **0.554 percentage points higher** than the validation rate. This is a difference in estimated risk between the two samples, not proof of better or worse model performance. Portfolio composition, the PD predictions, and the assumptions used for LGD and EAD all influence this comparison.

## 4. Loan-Level Risk Contribution

### 4.1 Largest individual expected-loss exposures

Each loan's contribution is calculated as its expected loss divided by total expected loss in the same portfolio. Ranking loans by absolute expected loss highlights the largest individual contributors.

In [8]:
train_df["el_contribution_pct"] = (
    train_df["expected_loss"] / train_df["expected_loss"].sum() * 100
)
valid_df["el_contribution_pct"] = (
    valid_df["expected_loss"] / valid_df["expected_loss"].sum() * 100
)

top_train_loans = train_df.nlargest(10, "expected_loss")
top_train_loans[[
    "id", "grade", "funded_amnt", "predicted_pd", "lgd_assumption",
    "expected_loss", "el_contribution_pct",
]].style.format({
    "funded_amnt": "${:,.2f}",
    "predicted_pd": "{:.2%}",
    "lgd_assumption": "{:.1%}",
    "expected_loss": "${:,.2f}",
    "el_contribution_pct": "{:.5f}%",
})

,id,grade,funded_amnt,predicted_pd,lgd_assumption,expected_loss,el_contribution_pct
1477742,117560607,G,"$35,000.00",100.00%,70.0%,"$24,499.80",0.00111%
1465326,113192566,E,"$40,000.00",89.74%,60.0%,"$21,538.44",0.00098%
1448840,109981518,F,"$35,000.00",100.00%,60.0%,"$20,999.81",0.00095%
1256594,94393614,E,"$35,000.00",99.94%,60.0%,"$20,987.11",0.00095%
1441171,109319640,E,"$35,000.00",99.83%,60.0%,"$20,964.43",0.00095%
1034428,77293666,F,"$35,000.00",99.74%,60.0%,"$20,946.30",0.00095%
1218818,91974215,C,"$40,000.00",100.00%,50.0%,"$19,999.12",0.00091%
1486807,113845057,C,"$40,000.00",99.96%,50.0%,"$19,991.34",0.00091%
1372805,104317572,E,"$35,000.00",94.23%,60.0%,"$19,787.50",0.00090%
1429452,110360482,F,"$40,000.00",77.43%,60.0%,"$18,582.86",0.00084%


**Interpretation**

The ten loans shown here are the highest individual contributors by **absolute expected loss**, not the highest contributors by percentage of the whole portfolio. In the current development output, the largest loan-level expected loss is approximately **$24,500**, or about **0.00111%** of total development expected loss. Its predicted PD is 100% under the model output. Extremely high PD predictions should be examined alongside PD-model calibration and prediction diagnostics in Notebook 02; they should not be interpreted as realized defaults.

### 4.2 Concentration among the largest contributors

The following table measures the share of total development expected loss attributable to the top 10, 100, and 1,000 loans. These figures describe concentration in the modelled expected-loss measure, not unexpected-loss risk or a complete concentration-risk framework.

In [9]:
total_train_el = train_df["expected_loss"].sum()
concentration_rows = []

for n in (10, 100, 1000):
    top_n_el = train_df.nlargest(n, "expected_loss")["expected_loss"].sum()
    concentration_rows.append({
        "Largest Loans": f"Top {n:,}",
        "Expected Loss ($)": top_n_el,
        "Share of Portfolio EL (%)": top_n_el / total_train_el * 100,
    })

concentration_df = pd.DataFrame(concentration_rows).set_index("Largest Loans")
concentration_df.style.format({
    "Expected Loss ($)": "${:,.2f}",
    "Share of Portfolio EL (%)": "{:.3f}%",
})

,Expected Loss ($),Share of Portfolio EL (%)
Largest Loans,,
Top 10,"$208,296.72",0.009%
Top 100,"$1,673,829.46",0.076%
"Top 1,000","$13,863,397.42",0.630%


**Interpretation**

Use this table to determine whether expected loss is concentrated in a small tail of exposures or is broadly distributed across the portfolio. A larger top-N share indicates more concentration in the largest individual expected-loss estimates. The table does not establish concentration in borrower groups, connected obligors, industries, or correlated risk factors.

## 5. Segment-Level Risk Analysis

Portfolio totals can obscure differences among borrower groups. For each segment, the analysis reports loan count, total EAD proxy, total expected loss, EAD-weighted average PD, EAD-weighted average assumed LGD, expected-loss rate, and share of portfolio expected loss.

For example, weighted-average PD is calculated as:

$$\overline{PD}_{EAD} = \frac{\sum_i PD_i EAD_i}{\sum_i EAD_i}$$

Weighted-average LGD is calculated analogously. Since PD and LGD are averaged separately, their product need not equal the segment expected-loss rate exactly; the loss rate is calculated directly from total expected loss divided by total EAD.

In [10]:
def assign_risk_band(df):
    out = df.copy()
    out["risk_band"] = pd.cut(
        out["predicted_pd"],
        bins=[-np.inf, 0.05, 0.15, np.inf],
        labels=["Low Risk", "Medium Risk", "High Risk"],
        right=False,
    )
    return out

train_df = assign_risk_band(train_df)
valid_df = assign_risk_band(valid_df)

risk_band_counts = (
    train_df["risk_band"].value_counts()
    .reindex(["Low Risk", "Medium Risk", "High Risk"])
    .rename("Loan Count")
    .to_frame()
)
display(risk_band_counts.style.format({"Loan Count": "{:,.0f}"}))

,Loan Count
risk_band,
Low Risk,"91,579"
Medium Risk,"551,643"
High Risk,"851,796"


### 5.1 Credit grade

In [11]:
def summarize_segments(df, segment_col):
    # Vectorized EAD-weighted numerators avoid a Python apply per group.
    work = df.assign(
        _ead_weighted_pd=df["predicted_pd"] * df["ead_proxy"],
        _ead_weighted_lgd=df["lgd_assumption"] * df["ead_proxy"],
    )
    grouped = work.groupby(segment_col, observed=True, dropna=False, sort=False)
    summary = grouped.agg(
        loan_count=("id", "size"),
        total_ead=("ead_proxy", "sum"),
        total_expected_loss=("expected_loss", "sum"),
        ead_weighted_pd_sum=("_ead_weighted_pd", "sum"),
        ead_weighted_lgd_sum=("_ead_weighted_lgd", "sum"),
    )
    summary["weighted_avg_pd"] = summary["ead_weighted_pd_sum"] / summary["total_ead"]
    summary["weighted_avg_lgd"] = summary["ead_weighted_lgd_sum"] / summary["total_ead"]
    summary["expected_loss_rate"] = summary["total_expected_loss"] / summary["total_ead"]
    summary["el_contribution_pct"] = (
        summary["total_expected_loss"] / df["expected_loss"].sum() * 100
    )
    summary = summary.drop(columns=["ead_weighted_pd_sum", "ead_weighted_lgd_sum"])
    return summary.sort_values("total_expected_loss", ascending=False)


def display_segment_summary(summary):
    display(summary.style.format({
        "loan_count": "{:,.0f}",
        "total_ead": "${:,.0f}",
        "total_expected_loss": "${:,.0f}",
        "weighted_avg_pd": "{:.2%}",
        "weighted_avg_lgd": "{:.2%}",
        "expected_loss_rate": "{:.2%}",
        "el_contribution_pct": "{:.2f}%",
    }))

grade_summary = summarize_segments(train_df, "grade")
display_segment_summary(grade_summary)

,loan_count,total_ead,total_expected_loss,weighted_avg_pd,weighted_avg_lgd,expected_loss_rate,el_contribution_pct
grade,,,,,,,
C,"432,460","$6,164,761,900","$673,356,583",21.85%,50.00%,10.92%,30.59%
D,"215,979","$3,329,133,450","$493,992,755",29.68%,50.00%,14.84%,22.44%
E,"101,745","$1,800,325,525","$405,375,563",37.53%,60.00%,22.52%,18.42%
B,"445,033","$5,883,519,125","$303,156,101",12.88%,40.00%,5.15%,13.77%
F,"34,302","$651,033,050","$173,908,578",44.52%,60.00%,26.71%,7.90%
A,"255,941","$3,573,022,225","$84,459,203",5.91%,40.00%,2.36%,3.84%
G,"9,558","$193,328,600","$66,719,794",49.30%,70.00%,34.51%,3.03%


**Interpretation — Credit grade**

Grade **C** is the largest contributor to development expected loss at approximately **$673.4 million (30.59%)**. Grades C, D, and E together contribute approximately **71.45%** of expected loss. Grade **G** has the highest segment expected-loss rate at about **34.51%**, while its share of total expected loss is much smaller at **3.03%**. This is the difference between a segment's loss intensity and its absolute contribution to a portfolio.

### 5.2 Loan purpose

In [12]:
purpose_summary = summarize_segments(train_df, "purpose")
display_segment_summary(purpose_summary)

,loan_count,total_ead,total_expected_loss,weighted_avg_pd,weighted_avg_lgd,expected_loss_rate,el_contribution_pct
purpose,,,,,,,
debt_consolidation,"867,320","$13,253,898,875","$1,443,087,162",21.43%,47.47%,10.89%,65.57%
credit_card,"331,393","$4,949,916,075","$404,609,114",17.01%,44.86%,8.17%,18.38%
home_improvement,"97,542","$1,365,451,825","$127,697,933",18.47%,46.78%,9.35%,5.80%
other,"85,033","$819,726,575","$89,724,644",20.80%,49.01%,10.95%,4.08%
small_business,"17,142","$262,928,175","$41,393,090",29.08%,50.89%,15.74%,1.88%
major_purchase,"32,307","$372,578,775","$36,898,231",19.47%,46.89%,9.90%,1.68%
medical,"17,058","$150,420,325","$16,399,948",21.20%,48.03%,10.90%,0.75%
house,"6,565","$97,890,875","$11,841,863",22.13%,50.58%,12.10%,0.54%
car,"16,179","$142,128,275","$10,420,019",14.79%,45.66%,7.33%,0.47%


**Interpretation — Loan purpose**

Debt consolidation contributes approximately **$1.443 billion (65.57%)** of development expected loss, followed by credit card loans at **18.38%**. Together these two purposes contribute about **83.95%** of total expected loss. Debt consolidation's leading contribution should be read alongside its much larger exposure: the purpose with the largest dollar loss is not automatically the purpose with the highest loss rate. In the current output, small-business loans have the highest purpose-level expected-loss rate at approximately **15.74%**, but represent a relatively small share of total expected loss.

### 5.3 PD-based risk band

In [13]:
risk_summary = summarize_segments(train_df, "risk_band")
display_segment_summary(risk_summary)

,loan_count,total_ead,total_expected_loss,weighted_avg_pd,weighted_avg_lgd,expected_loss_rate,el_contribution_pct
risk_band,,,,,,,
High Risk,"851,796","$13,177,783,625","$1,884,868,800",27.18%,51.19%,14.30%,85.64%
Medium Risk,"551,643","$7,336,411,400","$298,050,907",10.05%,40.32%,4.06%,13.54%
Low Risk,"91,579","$1,080,928,850","$18,048,869",4.17%,40.01%,1.67%,0.82%


**Interpretation — PD-based risk band**

The high-risk band contributes approximately **85.64%** of portfolio expected loss and has an expected-loss rate of **14.30%**. Medium risk contributes **13.54%** at a **4.06%** loss rate, while low risk contributes **0.82%** at a **1.67%** loss rate. These differences are consistent with the bands being defined from predicted PD thresholds (<5%, 5% to <15%, and ≥15%). They demonstrate how the current PD estimates distribute modelled loss, but do not independently prove that the PD model is well calibrated or predictive.

### 5.4 Borrower income band

In [14]:
def assign_income_band(df):
    out = df.copy()
    # pd.cut uses right-inclusive intervals here; labels make the boundaries explicit.
    out["income_band"] = pd.cut(
        out["annual_inc"],
        bins=[-np.inf, 40000, 80000, 120000, np.inf],
        labels=["≤ $40K", ">$40K to $80K", ">$80K to $120K", ">$120K"],
        right=True,
    ).astype("object")
    out["income_band"] = out["income_band"].fillna("Missing Income")
    return out

train_df = assign_income_band(train_df)
income_summary = summarize_segments(train_df, "income_band")
display_segment_summary(income_summary)

,loan_count,total_ead,total_expected_loss,weighted_avg_pd,weighted_avg_lgd,expected_loss_rate,el_contribution_pct
income_band,,,,,,,
>$40K to $80K,"735,268","$9,789,979,000","$1,057,325,680",21.30%,47.35%,10.80%,48.04%
>$80K to $120K,"319,852","$5,831,547,475","$581,481,106",19.78%,46.71%,9.97%,26.42%
>$120K,"168,233","$3,744,056,575","$307,423,172",16.58%,45.73%,8.21%,13.97%
≤ $40K,"271,661","$2,229,515,625","$254,738,160",22.65%,47.72%,11.43%,11.57%
Missing Income,4,"$25,200",$458,4.55%,40.00%,1.82%,0.00%


**Interpretation — Borrower income**

The **>$40K to $80K** segment contributes the largest share of development expected loss at approximately **$1.057 billion (48.04%)**. Among groups with reported income, the **≤$40K** segment has the highest expected-loss rate at about **11.43%**, while the **>$120K** segment has the lowest rate at **8.21%**. The four loans with missing income have only about **$458** of expected loss; they remain in the portfolio and are shown as a separate segment rather than being dropped.

### 5.5 Segment reconciliation checks

Each segmentation should account for the full development portfolio. These checks verify that the segment totals reconcile to the portfolio totals and that each segmentation's contribution percentages sum to approximately 100%.

In [15]:
segment_summaries = {
    "grade": grade_summary,
    "purpose": purpose_summary,
    "risk band": risk_summary,
    "income band": income_summary,
}

portfolio_ead = train_df["ead_proxy"].sum()
portfolio_el = train_df["expected_loss"].sum()

for label, summary in segment_summaries.items():
    assert np.isclose(summary["total_ead"].sum(), portfolio_ead, rtol=1e-10)
    assert np.isclose(summary["total_expected_loss"].sum(), portfolio_el, rtol=1e-10)
    assert np.isclose(summary["el_contribution_pct"].sum(), 100.0, atol=1e-8)
    print(f"{label.title()} segmentation reconciles to the portfolio.")

print("All segment reconciliation checks passed.")

Grade segmentation reconciles to the portfolio.
Purpose segmentation reconciles to the portfolio.
Risk Band segmentation reconciles to the portfolio.
Income Band segmentation reconciles to the portfolio.
All segment reconciliation checks passed.


## 6. Key Findings and Limitations

### 6.1 Key findings

- **Portfolio level:** Development expected loss is approximately **$2.201 billion** on **$21.595 billion** of EAD proxy, a **10.192%** expected-loss rate. Validation expected loss is approximately **$535.59 million** on **$5.557 billion** of EAD proxy, a **9.638%** rate.
- **Credit grade:** Grades C, D, and E together account for approximately **71.45%** of development expected loss. Grade G has the highest grade-level expected-loss rate but contributes a smaller share of total loss.
- **Loan purpose:** Debt consolidation and credit card loans together account for approximately **83.95%** of development expected loss.
- **PD-based risk band:** The high-risk band accounts for approximately **85.64%** of development expected loss. Because these bands are defined using PD thresholds, this is a descriptive segmentation result, not a standalone validation metric.
- **Income:** The >$40K to $80K segment contributes the most expected loss (**48.04%**); the ≤$40K band has the highest expected-loss rate among reported-income groups (**11.43%**).

### 6.2 Methodological limitations

- **PD model:** EL depends on the quality and calibration of the predicted PDs. Very high predictions, including predictions near 100%, should be examined in Notebook 02.
- **LGD:** LGD values are assigned assumptions by credit grade; they are not estimated from loan-level recovery outcomes in this notebook.
- **EAD:** Funded amount is used as an EAD proxy and does not reflect the actual outstanding balance at default.
- **Sample comparison:** Development and validation loss-rate differences can reflect sample composition and predicted risk; they are not, by themselves, evidence of model performance differences.
- **Expected versus realized loss:** These are model-based expected-loss estimates under the stated assumptions, not realized losses or a complete regulatory capital measure.
- **Segment boundaries:** Income cutoffs and PD bands are analytical choices for this project, not regulatory classifications.

### 6.3 Conclusion

Notebook 04 carries the PD/LGD/EAD components into an interpretable portfolio expected-loss framework. It verifies the loan-level calculations, quantifies aggregate loss rates, and identifies the grades, purposes, risk bands, and income segments associated with the largest modelled losses. Notebook 05 can build on these outputs for portfolio risk and stress testing, while retaining the limitations around assumed LGD and proxied EAD.